# Notebook 1 — Fetch Wikipedia Articles

This is the **first step** of the GraphRAG pipeline. It downloads plain-text Wikipedia articles for 51 famous scientists and saves each one as a `.txt` file in `data/wikipedia/`.

## What this notebook does
1. Reads Wikipedia page URLs from `data/scientist_wikipedia_urls.txt`
2. Fetches each article's full text via the **MediaWiki API** (no HTML parsing needed)
3. Lightly cleans the text (removes excessive blank lines)
4. Saves the result to `data/wikipedia/<PageTitle>.txt`

Already-downloaded files are skipped automatically, so the notebook is safe to re-run.

## Output
`data/wikipedia/` — 51 plain-text files, one per scientist (e.g. `Marie_Curie.txt`, `Alan_Turing.txt`)

## Next step
Once all articles are downloaded, run **Notebook 2** to extract named entities and relationships from the text using Azure OpenAI.

## 1 · Imports

In [ ]:
import time
import re
import requests
from pathlib import Path

## 2 · Configuration

In [ ]:
SCRIPT_DIR = Path(".").resolve()

URLS_FILE  = SCRIPT_DIR / "data" / "scientist_wikipedia_urls.txt"
OUTPUT_DIR = SCRIPT_DIR / "data" / "wikipedia"

MEDIAWIKI_API = "https://en.wikipedia.org/w/api.php"
REQUEST_DELAY = 1.0  # seconds between requests — be polite to Wikipedia

print(f"URLs file  : {URLS_FILE}")
print(f"Output dir : {OUTPUT_DIR}")

## 3 · Helper Functions

In [ ]:
def extract_title_from_url(url: str) -> str:
    """Extract the Wikipedia page title from a URL."""
    return url.rstrip("/").split("/wiki/")[-1]


def fetch_plain_text(title: str) -> str | None:
    """
    Fetch the full plain-text content of a Wikipedia page via the MediaWiki API.
    Returns None on failure.
    """
    params = {
        "action":      "query",
        "prop":        "extracts",
        "explaintext": True,
        "titles":      title,
        "format":      "json",
        "redirects":   1,
    }

    response = requests.get(
        MEDIAWIKI_API,
        params=params,
        headers={"User-Agent": "GraphRAG-Demo/1.0 (educational use)"},
        timeout=15,
    )
    response.raise_for_status()

    data  = response.json()
    pages = data.get("query", {}).get("pages", {})

    for page in pages.values():
        if "missing" in page:
            return None
        return page.get("extract", "")

    return None


def clean_text(text: str) -> str:
    """Remove excessive blank lines and leading/trailing whitespace."""
    # Collapse 3+ consecutive newlines into 2
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

## 4 · Fetch & Save Articles

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(URLS_FILE, encoding="utf-8") as f:
    urls = [line.strip() for line in f if line.strip()]

print(f"Processing {len(urls)} Wikipedia URLs...\n")

for url in urls:
    title    = extract_title_from_url(url)
    out_file = OUTPUT_DIR / f"{title}.txt"

    if out_file.exists():
        print(f"  [skip]  {title} (already downloaded)")
        continue

    print(f"  [fetch] {title}")
    try:
        text = fetch_plain_text(title)
        if text is None:
            print(f"  [warn]  Page not found: {title}")
            continue

        cleaned = clean_text(text)
        out_file.write_text(cleaned, encoding="utf-8")
        print(f"  [saved] {out_file.name} ({len(cleaned):,} chars)")

    except requests.RequestException as e:
        print(f"  [error] {title}: {e}")

    time.sleep(REQUEST_DELAY)

print("\nDone.")